# Notebook 1 — CNN Branch on CIFAR-10 (Feature Set A)

This notebook trains a Convolutional Neural Network (CNN) on **CIFAR-10 (Feature Set A)** from scratch.
It outputs fixed-length **"expertise embeddings"** (128-dimensional penultimate features) and full softmax probabilities for downstream Reinforcement Learning fusion in Notebook 3 (`03_fusion_rl.ipynb`).

### Training Optimizations Included:
- **Optimizer**: `AdamW` with decoupled weight decay ($10^{-2}$).
- **Learning Rate Scheduler**: `CosineAnnealingLR` decaying smoothly over 50 epochs.
- **Data Augmentation**: Standard CIFAR-10 random crop (padding=4) and random horizontal flip.
- **Model Checkpointing**: Automatically tracks validation accuracy and saves the best model (`best_cnn_checkpoint.pt`).

### Strict Data-Split Discipline (Reviewer Non-Negotiable)
To ensure complete mathematical integrity and prevent data leakage, CIFAR-10 is partitioned into **four independent splits**:
1. **`train_ds` (40,000 images)**: Optimized via gradient descent with data augmentations.
2. **`monitor_ds` (5,000 images)**: Clean unaugmented evaluation set used exclusively to track performance across epochs and save `best_cnn_checkpoint.pt`.
3. **`val_ds` (5,000 images)**: **Completely isolated and untouched during training.** Reserved exclusively to calibrate the RL fusion weights in Notebook 3 (`cnn_val_export.pt`).
4. **`test_ds` (10,000 images)**: Standard official CIFAR-10 test set, reserved strictly for the final reported evaluation in Notebook 3 (`cnn_test_export.pt`).

### Shared I/O Contract
Exports `cnn_val_export.pt` and `cnn_test_export.pt` with exact keys:
- `"embedding"`: FloatTensor `(N, EMBED_DIM)`
- `"probs"`: FloatTensor `(N, NUM_CLASSES)` (softmax output — drives the RL reward)
- `"pred"`: LongTensor `(N,)`
- `"label"`: LongTensor `(N,)`
- `"correct"`: IntTensor `(N,)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import os

# Set seeds for exact reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device Configuration (Automatically uses CUDA on Kaggle GPU / Colab, or CPU locally)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training & Architecture Hyperparameters
NUM_CLASSES = 10
EMBED_DIM = 128          # Penultimate expertise feature dimension (matches ViT and Notebook 3)
BATCH_SIZE = 128         # Standard batch size for CIFAR-10
NUM_EPOCHS = 50          # 50 epochs with Cosine Annealing
LR = 1e-3                # Initial learning rate for AdamW
WEIGHT_DECAY = 1e-2      # L2 regularization

# Hardware / Subset toggle:
# Set to None for full CIFAR-10 (40k train, 5k monitor, 5k val, 10k test).
# Set to e.g. 5000 if testing locally on a laptop CPU.
SUBSET_SIZE = None

print(f"Device: {DEVICE} | Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LR}")

## 1. CIFAR-10 Data Loading & 4-Way Splitting

We apply standard CIFAR-10 normalization (`mean=(0.4914, 0.4822, 0.4465)`, `std=(0.2470, 0.2435, 0.2616)`).
- Data augmentations (`RandomCrop`, `RandomHorizontalFlip`) are applied **only to the training split**.
- The `monitor`, `val`, and `test` splits use deterministic normalization without stochastic crops or flips.
- The random split indices use a fixed generator seed (`seed=42`) so the ViT notebook splits the exact same images.

In [ ]:
# Custom wrapper to apply distinct transforms to dataset subsets
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, target = self.dataset[self.indices[idx]]
        if self.transform is not None:
            img = self.transform(img)
        return img, target

# Standard CIFAR-10 transforms
norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

# Download base datasets
raw_cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
raw_cifar_test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

# Deterministic 4-way split: 40k train / 5k monitor / 5k RL val / 10k test
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(raw_cifar_train), generator=g).tolist()

if SUBSET_SIZE is not None:
    n_tr, n_mo, n_va = int(SUBSET_SIZE * 0.8), int(SUBSET_SIZE * 0.1), int(SUBSET_SIZE * 0.1)
else:
    n_tr, n_mo, n_va = 40000, 5000, 5000

train_idx = indices[:n_tr]
monitor_idx = indices[n_tr:n_tr + n_mo]
val_idx = indices[n_tr + n_mo:n_tr + n_mo + n_va]

train_ds = TransformedSubset(raw_cifar_train, train_idx, train_transform)
monitor_ds = TransformedSubset(raw_cifar_train, monitor_idx, eval_transform)
val_ds = TransformedSubset(raw_cifar_train, val_idx, eval_transform)
test_ds = TransformedSubset(raw_cifar_test, list(range(len(raw_cifar_test))), eval_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE=="cuda"))
monitor_dl = DataLoader(monitor_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

print(f"Split Sizes: Train={len(train_ds)}, Monitor={len(monitor_ds)}, RL Val={len(val_ds)}, Test={len(test_ds)}")

## 2. CNN Architecture

A 3-stage ConvNet tailored for CIFAR-10 with batch normalization, max pooling, and global average pooling.
`forward(x, return_embedding=True)` produces the 128-dimensional penultimate **expertise feature** representing spatial/local inductive biases.

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=EMBED_DIM):
        super().__init__()
        self.features = nn.Sequential(
            # Stage 1: 32x32 -> 16x16
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Stage 2: 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Stage 3: 8x8 -> 1x1
            nn.Conv2d(128, embed_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_embedding=False):
        feat = self.features(x).flatten(1)   # (B, embed_dim) — penultimate expertise feature
        logits = self.classifier(feat)
        if return_embedding:
            return logits, feat
        return logits

model = BasicCNN().to(DEVICE)
print(model)

## 3. Training Loop with AdamW, Cosine Annealing & Checkpointing

- Optimizes cross-entropy loss with **AdamW**.
- Steps **CosineAnnealingLR** at the end of each epoch to smoothly anneal the learning rate from $10^{-3}$ down to $10^{-5}$.
- Evaluates on `monitor_dl` after every epoch to track accuracy and save the best checkpoint (`best_cnn_checkpoint.pt`).
- **Split discipline preserved**: The RL `val` split is never used here.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=1e-5)

def evaluate(dl):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss_sum += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total

best_monitor_acc = 0.0
checkpoint_path = "best_cnn_checkpoint.pt"

print(f"--- Starting Training for {NUM_EPOCHS} Epochs ---")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt.step()

        tr_loss += loss.item() * len(yb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total += len(yb)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    train_acc = tr_correct / tr_total
    train_loss = tr_loss / tr_total

    # Evaluate on the independent monitor split
    mo_loss, mo_acc = evaluate(monitor_dl)

    # Checkpoint saving on best monitor accuracy
    if mo_acc > best_monitor_acc:
        best_monitor_acc = mo_acc
        torch.save(model.state_dict(), checkpoint_path)
        saved_marker = "* (Saved Best)"
    else:
        saved_marker = ""

    if epoch % 5 == 0 or epoch == 1 or epoch == NUM_EPOCHS or saved_marker != "":
        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Monitor Acc: {mo_acc:.3f} (Best: {best_monitor_acc:.3f}) {saved_marker}")

print(f"Training complete! Best Monitor Accuracy: {best_monitor_acc:.4f}")

## 4. Reload Best Checkpoint & Export (Val & Test)

Before saving the exports for Notebook 3, we reload the **best checkpoint weights** (`best_cnn_checkpoint.pt`).
We run inference on:
1. `val_dl` (5,000 samples) -> saved as `cnn_val_export.pt`
2. `test_dl` (10,000 samples) -> saved as `cnn_test_export.pt`

In [ ]:
# Load best checkpoint weights
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print(f"Loaded best model weights from {checkpoint_path}")

@torch.no_grad()
def export_split(dl):
    model.eval()
    embs, probs, preds, labels, correct = [], [], [], [], []
    for xb, yb in dl:
        xb = xb.to(DEVICE)
        logits, feat = model(xb, return_embedding=True)
        pr = F.softmax(logits, dim=1).cpu()
        p = pr.argmax(1)
        embs.append(feat.cpu())
        probs.append(pr)
        preds.append(p)
        labels.append(yb.cpu())
        correct.append((p == yb.cpu()).int())
    return (
        torch.cat(embs),
        torch.cat(probs),
        torch.cat(preds),
        torch.cat(labels),
        torch.cat(correct),
    )

# 1. Export VAL split (for RL calibration in Notebook 3)
val_emb, val_probs, val_pred, val_label, val_correct = export_split(val_dl)
torch.save({
    "embedding": val_emb,
    "probs": val_probs,
    "pred": val_pred,
    "label": val_label,
    "correct": val_correct,
}, "cnn_val_export.pt")
print("Saved cnn_val_export.pt:", val_emb.shape, "val_acc=", val_correct.float().mean().item())

# 2. Export TEST split (for final one-time evaluation in Notebook 3)
test_emb, test_probs, test_pred, test_label, test_correct = export_split(test_dl)
torch.save({
    "embedding": test_emb,
    "probs": test_probs,
    "pred": test_pred,
    "label": test_label,
    "correct": test_correct,
}, "cnn_test_export.pt")
print("Saved cnn_test_export.pt:", test_emb.shape, "test_acc=", test_correct.float().mean().item())